In [1]:
import pandas as pd

X_train_balanced = pd.read_csv("processed/X_train_balanced.csv")
y_train_balanced = pd.read_csv(
    "processed/y_train_balanced.csv"
).squeeze()

print("X_train_balanced:", X_train_balanced.shape)
print("y_train_balanced:", y_train_balanced.shape)

print("\nClass distribution:")
print(y_train_balanced.value_counts())

X_train_balanced: (788, 30)
y_train_balanced: (788,)

Class distribution:
Class
0    394
1    394
Name: count, dtype: int64


In [2]:
from sklearn.model_selection import GridSearchCV, StratifiedKFold

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

In [3]:
from sklearn.linear_model import LogisticRegression

lr = LogisticRegression(
    max_iter=2000,
    random_state=42
)

lr_params = {
    "C": [0.01, 0.1, 1, 10, 100],
    "solver": ["liblinear", "lbfgs"]
}

lr_grid = GridSearchCV(
    estimator=lr,
    param_grid=lr_params,
    cv=cv,
    scoring="roc_auc",
    n_jobs=-1
)

lr_grid.fit(X_train_balanced, y_train_balanced)

print("Best parameters:")
print(lr_grid.best_params_)

print("Best CV ROC-AUC:")
print(lr_grid.best_score_)

Best parameters:
{'C': 0.01, 'solver': 'lbfgs'}
Best CV ROC-AUC:
0.9798372220099509


In [4]:
from sklearn.tree import DecisionTreeClassifier

dt = DecisionTreeClassifier(
    random_state=42
)

dt_params = {
    "max_depth": [3, 5, 10, 15, 20, None],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 5, 10],
    "criterion": ["gini", "entropy"]
}

dt_grid = GridSearchCV(
    estimator=dt,
    param_grid=dt_params,
    cv=cv,
    scoring="roc_auc",
    n_jobs=-1
)

dt_grid.fit(X_train_balanced, y_train_balanced)

print("Best parameters:", dt_grid.best_params_)
print("Best CV ROC-AUC:", dt_grid.best_score_)

Best parameters: {'criterion': 'gini', 'max_depth': 10, 'min_samples_leaf': 10, 'min_samples_split': 2}
Best CV ROC-AUC: 0.9634649279577978


In [5]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(
    random_state=42,
    n_jobs=-1
)

rf_params = {
    "n_estimators": [100, 200],
    "max_depth": [None, 5, 10, 20],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 5],
    "max_features": ["sqrt", "log2"]
}

rf_grid = GridSearchCV(
    estimator=rf,
    param_grid=rf_params,
    cv=cv,
    scoring="roc_auc",
    n_jobs=-1
)

rf_grid.fit(X_train_balanced, y_train_balanced)

print("Best parameters:", rf_grid.best_params_)
print("Best CV ROC-AUC:", rf_grid.best_score_)

Best parameters: {'max_depth': None, 'max_features': 'log2', 'min_samples_leaf': 2, 'min_samples_split': 5, 'n_estimators': 100}
Best CV ROC-AUC: 0.982295736629978


In [6]:
from sklearn.neighbors import KNeighborsClassifier

knn = KNeighborsClassifier()

knn_params = {
    "n_neighbors": [3, 5, 7, 9, 11, 15],
    "weights": ["uniform", "distance"],
    "metric": ["euclidean", "manhattan"]
}

knn_grid = GridSearchCV(
    estimator=knn,
    param_grid=knn_params,
    cv=cv,
    scoring="roc_auc",
    n_jobs=-1
)

knn_grid.fit(X_train_balanced, y_train_balanced)

print("Best parameters:", knn_grid.best_params_)
print("Best CV ROC-AUC:", knn_grid.best_score_)

Best parameters: {'metric': 'manhattan', 'n_neighbors': 15, 'weights': 'distance'}
Best CV ROC-AUC: 0.9734772944835435


In [7]:
from sklearn.svm import SVC

svm = SVC(
    probability=True,
    random_state=42
)

svm_params = {
    "C": [0.1, 1, 10],
    "kernel": ["linear", "rbf"],
    "gamma": ["scale", "auto"]
}

svm_grid = GridSearchCV(
    estimator=svm,
    param_grid=svm_params,
    cv=cv,
    scoring="roc_auc",
    n_jobs=-1
)

svm_grid.fit(X_train_balanced, y_train_balanced)

print("Best parameters:", svm_grid.best_params_)
print("Best CV ROC-AUC:", svm_grid.best_score_)

Best parameters: {'C': 0.1, 'gamma': 'scale', 'kernel': 'rbf'}
Best CV ROC-AUC: 0.9817283554985847


In [9]:
tuned_models = {
    "Logistic Regression": lr_grid.best_estimator_,
    "Decision Tree": dt_grid.best_estimator_,
    "Random Forest": rf_grid.best_estimator_,
    "KNN": knn_grid.best_estimator_,
    "SVM": svm_grid.best_estimator_,
    "Naive Bayes": nb_grid.best_estimator_
}

In [10]:
import joblib
from pathlib import Path

Path("models").mkdir(exist_ok=True)

for name, model in tuned_models.items():

    filename = (
        name.lower()
        .replace(" ", "_")
    )

    joblib.dump(
        model,
        f"models/tuned_{filename}.pkl"
    )

print("All tuned models saved.")

All tuned models saved.


In [11]:
results = []

grids = {
    "Logistic Regression": lr_grid,
    "Decision Tree": dt_grid,
    "Random Forest": rf_grid,
    "KNN": knn_grid,
    "SVM": svm_grid,
    "Naive Bayes": nb_grid
}

for name, grid in grids.items():

    results.append({
        "Model": name,
        "Best CV ROC-AUC": grid.best_score_,
        "Best Parameters": grid.best_params_
    })

tuning_results = pd.DataFrame(results)

tuning_results

,Model,Best CV ROC-AUC,Best Parameters
0,Logistic Regression,0.979837,"{'C': 0.01, 'solver': 'lbfgs'}"
1,Decision Tree,0.963465,"{'criterion': 'gini', 'max_depth': 10, 'min_sa..."
2,Random Forest,0.982296,"{'max_depth': None, 'max_features': 'log2', 'm..."
3,KNN,0.973477,"{'metric': 'manhattan', 'n_neighbors': 15, 'we..."
4,SVM,0.981728,"{'C': 0.1, 'gamma': 'scale', 'kernel': 'rbf'}"
5,Naive Bayes,0.959784,{'var_smoothing': 1e-12}


In [12]:
from pathlib import Path

Path("results").mkdir(exist_ok=True)

tuning_results.to_csv(
    "results/hyperparameter_tuning_results.csv",
    index=False
)